In [9]:
import pandas as pd
import joblib
import os
from dotenv import load_dotenv
from conformist import FNRCoP, PredictionDataset, ValidationTrial

load_dotenv()

RANDOM_SEED = int(os.getenv("RANDOM_SEED", 42))
N_JOBS = int(os.getenv("N_JOBS", 10))
X_CALIBRATION_CSV_PATH = os.getenv("X_CALIBRATION_CSV_PATH")
Y_CALIBRATION_TSV_PATH = os.getenv("Y_CALIBRATION_TSV_PATH")
X_TEST_CSV_PATH = os.getenv("X_TEST_CSV_PATH")
Y_TEST_TSV_PATH = os.getenv("Y_TEST_TSV_PATH")
RF_MODEL_PATH = os.getenv("RF_MODEL_PATH")
SELECTED_FEATURES_PATH = os.getenv("SELECTED_FEATURES_PATH")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", ".")

# Print config
print(f"RANDOM_SEED: {RANDOM_SEED}")
print(f"N_JOBS: {N_JOBS}")
print(f"X_CALIBRATION_CSV_PATH: {X_CALIBRATION_CSV_PATH}")
print(f"Y_CALIBRATION_TSV_PATH: {Y_CALIBRATION_TSV_PATH}")
print(f"X_TEST_CSV_PATH: {X_TEST_CSV_PATH}")
print(f"Y_TEST_TSV_PATH: {Y_TEST_TSV_PATH}")
print(f"RF_MODEL_PATH: {RF_MODEL_PATH}")
print(f"SELECTED_FEATURES_PATH: {SELECTED_FEATURES_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

RANDOM_SEED: 42
N_JOBS: 10
X_CALIBRATION_CSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
Y_CALIBRATION_TSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
X_TEST_CSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-test/transpose/transposed.csv
Y_TEST_TSV_PATH: /home/mariya/Data/pallas/nf-core-gexml-output-test/class/subtypes.tsv
RF_MODEL_PATH: /home/mariya/Data/pallas/training/final_model.pkl
SELECTED_FEATURES_PATH: /home/mariya/Data/pallas/training/selected_features.pkl
OUTPUT_DIR: /home/mariya/Data/pallas/conformal_prediction


In [2]:
# Load the model
rf = joblib.load(RF_MODEL_PATH)

# Get probabilities for calibration set
X_calibration = pd.read_csv(X_CALIBRATION_CSV_PATH, index_col=0)
y_calibration = pd.read_csv(Y_CALIBRATION_TSV_PATH, index_col=0, sep="\t").squeeze()
X_calibration.shape, y_calibration.shape

((459, 19916), (459,))

In [3]:
# Load selected features
selected_features = joblib.load(SELECTED_FEATURES_PATH)
X_calibration = X_calibration[selected_features]
X_calibration.shape

(459, 3931)

In [4]:
# Align X and y
X_calibration = X_calibration.loc[y_calibration.index]
X_calibration, y_calibration

(          ENSG00000000938  ENSG00000001036  ENSG00000001561  ENSG00000001617  \
 sample                                                                         
 GE8315           3.121406        -1.335113         4.097582         7.298677   
 ALLT-347         4.504943         3.283844         3.204502         2.992957   
 GE2328           5.580783         4.396877         5.464106         6.451601   
 ALLT-368         6.443963         2.580803         3.977706         4.467344   
 ALLT-342         6.217555         2.972139         6.157332         4.338325   
 ...                   ...              ...              ...              ...   
 ALL_200          2.329466         2.546031         3.217649         3.774056   
 ALL_351          2.154335         0.318302         1.806234         1.873627   
 ALL_626          1.305108        -1.335113         0.633883         2.970439   
 ALL_210          4.023584         2.813499         3.239340         3.116271   
 ALL_254          1.470705  

In [8]:
probs_calibration = rf.predict_proba(X_calibration)
preds_calibration = rf.predict(X_calibration)

df = pd.DataFrame(index=X_calibration.index)

# Add a column with known_class
df["known_class"] = y_calibration

# Add a column with the predicted class
df["predicted_class"] = preds_calibration

probs_df = pd.DataFrame(probs_calibration, columns=rf.classes_, index=X_calibration.index)
df = pd.concat([df, probs_df], axis=1)

filename = "gex_calibration_probs.csv"

# Rename index to "id"
df = df.rename_axis("id")

# Save
df.to_csv(os.path.join(OUTPUT_DIR, filename))
df

,known_class,predicted_class,"AML, APL-PML::RARA","AML, CBFA2T3::GLIS2","AML, CBFB::MYH11","AML, CEBPAalt","AML, KMT2A-r","AML, NPM1alt","AML, NUP98-r","AML, RUNX1::RUNX1T1",...,"B-ALL, NUTM1-r","B-ALL, PAX5 P80R","B-ALL, PAX5alt","B-ALL, TCF3::PBX1","B-ALL, ZNF384-r","B-ALL, high hyperdiploid","B-ALL, hypodiploid","B-ALL, iAMP21",Non-leukemic,T-ALL
id,,,,,,,,,,,,,,,,,,,,,
GE8315,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.001841,0.003910,0.002044,0.004635,0.005458,0.000918,0.001724,0.002949,...,0.005851,0.000000,0.011116,0.010588,0.009749,0.021401,0.018793,0.014695,0.000000,0.012381
ALLT-347,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.001912,0.001843,0.004221,0.003592,0.010469,0.013180,0.005377,0.006755,...,0.005749,0.018622,0.044852,0.009389,0.011307,0.474949,0.125008,0.026358,0.001965,0.016576
GE2328,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.005694,0.005347,0.002643,0.003340,0.006792,0.003130,0.002724,0.016650,...,0.008305,0.003238,0.024898,0.011336,0.031164,0.033394,0.032694,0.019745,0.000000,0.026138
ALLT-368,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.001843,0.000000,0.001555,0.004922,0.000176,0.000918,0.001687,0.001532,...,0.010553,0.003397,0.023259,0.004513,0.008271,0.709865,0.065213,0.027240,0.001902,0.011505
ALLT-342,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.000000,0.001843,0.000930,0.000000,0.004852,0.000513,0.001986,0.002655,...,0.005596,0.009214,0.018730,0.008843,0.023159,0.024941,0.027737,0.022878,0.007027,0.013753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ALL_200,"B-ALL, DUX4-r","B-ALL, DUX4-r",0.003790,0.000000,0.002153,0.002965,0.004985,0.002697,0.000000,0.003034,...,0.000802,0.016119,0.028404,0.005944,0.044460,0.015205,0.029918,0.006943,0.000000,0.006158
ALL_351,"B-ALL, PAX5alt","B-ALL, PAX5alt",0.000000,0.003715,0.000999,0.001443,0.004771,0.001983,0.005958,0.003689,...,0.015271,0.044714,0.288712,0.016456,0.013385,0.100643,0.089940,0.063469,0.005641,0.033360
ALL_626,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.003873,0.005839,0.000000,0.000000,0.003229,0.005546,0.007387,0.004492,...,0.007913,0.032416,0.035567,0.007216,0.007954,0.588179,0.077108,0.037742,0.003873,0.011907
